# LeetCode 4033 — Valid K-Unique Subarrays I bằng Python

- Link: https://leetcode.com/problems/valid-k-unique-subarrays-i/
- Độ khó: **Hard**
- Chủ đề: Prefix XOR + hàm băm ngẫu nhiên (hashing), Cây Fenwick (BIT), Truy vấn offline, Đếm phần tử phân biệt trong đoạn

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def validSubarrays(self, nums: list[int], k: int, queries: list[list[int]]) -> list[bool]:
```

> Ghi chú: bản tham khảo (doocs) của bài này chưa có lời giải Python; các lời giải dưới đây do notebook tự xây và tự kiểm chứng bằng vét cạn.

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums`, số `k` và rất nhiều truy vấn `[l, r]`. Một đoạn con `nums[l..r]` là **hợp lệ** nếu (1) có **đúng `k`** giá trị phân biệt và (2) **mọi giá trị đều xuất hiện số lần chẵn**. Trả lời đúng/sai cho từng truy vấn.

Với `n, q` tới `10^5` ta không thể duyệt đoạn con cho mỗi truy vấn. Bài này là sự kết hợp của hai kỹ thuật độc lập:

- **(A) "Tất cả tần suất chẵn"** ⇒ kiểm tra bằng prefix XOR của một hàm băm ngẫu nhiên, mỗi truy vấn `O(1)`.
- **(B) "Đúng `k` giá trị phân biệt trong đoạn"** ⇒ đếm phần tử phân biệt trong đoạn bằng **Fenwick tree xử lý offline** theo đầu mút phải.

Sau notebook này bạn sẽ:

1. Hiểu vì sao `a ^ a = 0` biến bài toán "số lần chẵn" thành so sánh hai prefix.
2. Biết dùng hàm băm ngẫu nhiên để nén một "mặt nạ parity" khổng lồ xuống 64 bit, và hiểu xác suất sai.
3. Cài đặt Fenwick tree (BIT) và kỹ thuật "chỉ giữ lần xuất hiện cuối cùng" để đếm phần tử phân biệt trong đoạn.
4. Xử lý truy vấn **offline**: sắp xếp theo `r`, quét một lần.
5. Đối chiếu 4 cách (vét cạn, parity mặt nạ chính xác, hash + đếm thẳng, hash + BIT) bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Với mỗi truy vấn `[l, r]` (`0 <= l < r <= n - 1`), đoạn `nums[l..r]` hợp lệ nếu:

1. Nó có **đúng `k`** số phân biệt, và
2. **Tần suất của mọi số trong đoạn đều chẵn.**

Trả về mảng boolean `ans`.

**Ví dụ 1:** `nums = [1,2,2,1]`, `k = 2`, `queries = [[0,1],[0,3],[1,2]]` → `[false, true, false]`.

- `[0,1] = [1,2]`: tần suất `{1:1, 2:1}` lẻ → sai.
- `[0,3] = [1,2,2,1]`: `{1:2, 2:2}`, đúng 2 số phân biệt, đều chẵn → đúng.
- `[1,2] = [2,2]`: chỉ 1 số phân biệt (< k) → sai.

**Ví dụ 2:** `nums = [3,3,3]`, `k = 1`, `queries = [[1,2],[0,2]]` → `[true, false]`.

**Ràng buộc**

- `2 <= n == nums.length <= 10^5`
- `1 <= nums[i] <= 10^5`
- `1 <= k <= n`
- `1 <= queries.length <= 10^5`
- `queries[i] = [li, ri]`, `0 <= li < ri <= n - 1`

In [1]:
from typing import List
from collections import Counter

EXAMPLES = [
    ([1, 2, 2, 1], 2, [[0, 1], [0, 3], [1, 2]], [False, True, False]),
    ([3, 3, 3], 1, [[1, 2], [0, 2]], [True, False]),
]
for nums, k, qs, exp in EXAMPLES:
    print(f"nums={nums}, k={k}, queries={qs} -> kỳ vọng {exp}")

nums=[1, 2, 2, 1], k=2, queries=[[0, 1], [0, 3], [1, 2]] -> kỳ vọng [False, True, False]
nums=[3, 3, 3], k=1, queries=[[1, 2], [0, 2]] -> kỳ vọng [True, False]


## 3. Kiến thức nền

### 3.1. `Counter` và tần suất

`collections.Counter(list)` đếm số lần xuất hiện của mỗi phần tử. `len(counter)` là số phần tử phân biệt; `counter.values()` là các tần suất.

In [2]:
c = Counter([1, 2, 2, 1])
print(c, "| phân biệt:", len(c), "| mọi tần suất chẵn?", all(v % 2 == 0 for v in c.values()))
c = Counter([2, 2])
print(c, "| phân biệt:", len(c))

Counter({1: 2, 2: 2}) | phân biệt: 2 | mọi tần suất chẵn? True
Counter({2: 2}) | phân biệt: 1


### 3.2. Toán tử XOR `^` và tính chất `a ^ a = 0`

XOR hai số theo từng bit: `1 ^ 1 = 0`, `1 ^ 0 = 1`. Hai tính chất cần nhớ:

- `a ^ a = 0` và `a ^ 0 = a`.
- XOR có tính **giao hoán, kết hợp** ⇒ thứ tự không quan trọng.

Do đó XOR một giá trị `a` đủ **số lần chẵn** thì triệt tiêu, **số lần lẻ** thì còn lại `a`.

In [3]:
print(5 ^ 5, 5 ^ 0, 3 ^ 5 ^ 3)           # 0 5 5
print(7 ^ 7 ^ 7)                          # 7  (lẻ lần -> còn lại)
print(7 ^ 7 ^ 7 ^ 7)                      # 0  (chẵn lần -> hết)

0 5 5
7
0


### 3.3. Prefix XOR: XOR của một đoạn bằng hai prefix

`P[0] = 0`, `P[i+1] = P[i] ^ a[i]`. Khi đó XOR của đoạn `a[l..r]` là `P[r+1] ^ P[l]` (phần chung triệt tiêu).

In [4]:
a = [5, 9, 5, 12]
P = [0]
for x in a:
    P.append(P[-1] ^ x)
print("P =", P)
l, r = 1, 3
print("XOR a[1..3] bằng prefix:", P[r + 1] ^ P[l], "| tính thẳng:", 9 ^ 5 ^ 12)

P = [0, 5, 12, 9, 5]
XOR a[1..3] bằng prefix: 0 | tính thẳng: 0


### 3.4. Mặt nạ parity (bitmask) và ý tưởng băm

Muốn biết "mỗi giá trị xuất hiện chẵn hay lẻ trong đoạn", ta gán cho giá trị `v` một **bit** riêng `1 << v` rồi XOR dồn: bit `v` của prefix mask cho biết parity của số lần `v` đã xuất hiện. Đoạn `[l, r]` có mọi tần suất chẵn ⇔ `mask[r+1] == mask[l]`.

Vấn đề: `v` tới `10^5` nên mỗi mask dài `10^5` bit, `n` mask quá lớn. Ý tưởng **băm**: thay `1 << v` bằng một **số ngẫu nhiên 64 bit** `h[v]` gán cho mỗi giá trị. XOR vẫn triệt tiêu như cũ; hai mask chỉ "vô tình" bằng nhau với xác suất ≈ `2^-64`.

In [5]:
import random
rng = random.Random(123)
h = {}
def hv(v):
    if v not in h:
        h[v] = rng.getrandbits(64)
    return h[v]

nums = [1, 2, 2, 1, 3]
P = [0]
for x in nums:
    P.append(P[-1] ^ hv(x))
print([hex(p)[:10] for p in P])
print("đoạn [0,3] chẵn?", P[4] == P[0])    # [1,2,2,1] -> True
print("đoạn [0,4] chẵn?", P[5] == P[0])    # có số 3 lẻ -> False
print("đoạn [1,2] chẵn?", P[3] == P[1])    # [2,2] -> True

['0x0', '0x44867db3', '0x805c2ecf', '0x44867db3', '0x0', '0x443cdef3']
đoạn [0,3] chẵn? True
đoạn [0,4] chẵn? False
đoạn [1,2] chẵn? True


### 3.5. Cây Fenwick (BIT): cộng một điểm, hỏi tổng tiền tố

BIT lưu mảng đếm `tree` cho phép:

- `add(i, delta)`: cộng `delta` vào vị trí `i` trong `O(log n)`.
- `prefix(i)`: tổng các vị trí `0..i` trong `O(log n)`.

Mấu chốt là phép `i & -i` (lowbit): lấy bit 1 thấp nhất. Dùng chỉ số 1-based nội bộ.

In [6]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)          # 1-based

    def add(self, i, delta):            # i: 0-based
        i += 1
        while i <= self.n:
            self.t[i] += delta
            i += i & -i

    def prefix(self, i):                # tổng a[0..i], i = -1 -> 0
        i += 1
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

print([x & -x for x in range(1, 9)])    # lowbit: 1 2 1 4 1 2 1 8
bit = Fenwick(8)
for pos in [0, 3, 3, 7]:
    bit.add(pos, 1)
print([bit.prefix(i) for i in range(8)])  # tổng tích luỹ: 1 1 1 3 3 3 3 4
print("đếm trong [3, 7] =", bit.prefix(7) - bit.prefix(2))

[1, 2, 1, 4, 1, 2, 1, 8]
[1, 1, 1, 3, 3, 3, 3, 4]
đếm trong [3, 7] = 3


### 3.6. Kỹ thuật "đếm số phần tử phân biệt trong đoạn" (offline)

Duyệt `i = 0, 1, ..., n-1`. Duy trì một mảng `mark` mà `mark[j] = 1` **chỉ** nếu `j` là lần xuất hiện **cuối cùng (tính đến `i`)** của `nums[j]`. Khi gặp `nums[i]`:

- nếu nó đã xuất hiện ở `last = last_pos[nums[i]]` thì `mark[last] -= 1`;
- rồi `mark[i] += 1`.

Khi đó với truy vấn `[l, i]`, **số giá trị phân biệt** = tổng `mark[l..i]` — vì mỗi giá trị trong `[l, i]` có đúng một lần xuất hiện cuối cùng và nó nằm trong `[l, i]`. Dùng BIT cho tổng đoạn.

In [7]:
nums = [1, 2, 1, 3, 2]
bit = Fenwick(len(nums))
last = {}
for i, x in enumerate(nums):
    if x in last:
        bit.add(last[x], -1)
    bit.add(i, +1)
    last[x] = i
    print(f"i={i}, x={x}: mark = {[bit.prefix(j) - bit.prefix(j - 1) for j in range(len(nums))]}")
print("phân biệt trong nums[1..4]:", bit.prefix(4) - bit.prefix(0))   # 2,1,3,2 -> {1,2,3} = 3

i=0, x=1: mark = [1, 0, 0, 0, 0]
i=1, x=2: mark = [1, 1, 0, 0, 0]
i=2, x=1: mark = [0, 1, 1, 0, 0]
i=3, x=3: mark = [0, 1, 1, 1, 0]
i=4, x=2: mark = [0, 0, 1, 1, 1]
phân biệt trong nums[1..4]: 3


## 4. Ý tưởng nền tảng / trực giác

**Hai điều kiện độc lập.** Một đoạn hợp lệ ⇔ (A) mọi tần suất chẵn **và** (B) số giá trị phân biệt = `k`. Ta trả lời từng điều kiện bằng một kỹ thuật riêng rồi AND lại.

**Tính tay ví dụ 1.** `nums = [1,2,2,1]`, `k = 2`.

- Parity theo prefix (dùng mặt nạ bit, `1 << v`): `P = [0, 2, 6, 2, 0]` (bit 1 = giá trị 1 bật/tắt, bit 2 = giá trị 2).
- Truy vấn `[0,1]`: `P[2] = 6` vs `P[0] = 0` khác → lẻ → sai.
- Truy vấn `[0,3]`: `P[4] = 0 = P[0]` → chẵn; phân biệt `= 2 = k` → **đúng**.
- Truy vấn `[1,2]`: `P[3] = 2 = P[1]` → chẵn; phân biệt `= 1 ≠ 2` → sai.

**Chiến lược.** Với mỗi truy vấn: so hai prefix hash `O(1)`; chỉ khi chẵn mới cần đếm số phân biệt (ta sẽ đếm bằng BIT cho gọn và chắc chắn `O((n + q) log n)`).

## 5. Cách 1 — Vét cạn: đếm tần suất từng truy vấn

**Ý tưởng.** Với mỗi truy vấn, lấy lát `nums[l:r+1]`, đếm bằng `Counter`, kiểm tra `len(counter) == k` và mọi tần suất chẵn. Đây là bản thể hiện **nguyên văn đề bài**, ta dùng làm trọng tài.

In [8]:
def valid_subarrays_bruteforce(nums, k, queries):
    ans = []
    for l, r in queries:
        c = Counter(nums[l:r + 1])
        ans.append(len(c) == k and all(v % 2 == 0 for v in c.values()))
    return ans

for nums, k, qs, exp in EXAMPLES:
    print(valid_subarrays_bruteforce(nums, k, qs), exp)

[False, True, False] [False, True, False]
[True, False] [True, False]


**Độ phức tạp.** Mỗi truy vấn tốn `O(r - l + 1)` ⇒ tổng `O(q · n)` trường hợp xấu — với `n = q = 10^5` là `10^10`. **Quá chậm.**

**Nút thắt:** ta đếm lại cả đoạn cho mỗi truy vấn trong khi hai đại lượng cần biết (parity và số phân biệt) đều có thể **chuẩn bị trước**.

## 6. Bước đột phá

**Insight 1 — chẵn/lẻ là hiệu của hai prefix.** Với hàm băm `h` ngẫu nhiên trên giá trị, `P[i] = XOR h[nums[j]]` (`j < i`). Đoạn `[l, r]` có mọi tần suất chẵn ⇔ `P[r+1] == P[l]` (đúng chắc chắn một chiều; chiều ngược lại đúng với xác suất cao).

- Nếu mọi tần suất chẵn ⇒ các `h` triệt tiêu từng cặp ⇒ `P[r+1] ^ P[l] = 0` **luôn đúng**.
- Nếu có tập giá trị tần suất lẻ khác rỗng ⇒ `P[r+1] ^ P[l] = XOR h[v]` của tập đó; với `h` ngẫu nhiên 64 bit, xác suất bằng 0 là `2^-64` (với mỗi truy vấn).

**Insight 2 — đếm phân biệt trong đoạn offline.** Sắp xếp truy vấn theo `r`. Quét `i` từ trái sang phải, duy trì BIT đánh dấu "lần xuất hiện cuối của mỗi giá trị" (mục 3.6). Với truy vấn `r = i`, số phân biệt = `bit.prefix(r) - bit.prefix(l - 1)`.

**Quy tắc nếu… thì…**

- Nếu `P[r+1] != P[l]` ⇒ chắc chắn sai (có giá trị lẻ).
- Nếu `P[r+1] == P[l]` và `distinct == k` ⇒ đúng.
- Ngược lại ⇒ sai.

**Bất biến BIT.** Sau khi xử lý chỉ số `i`, `bit` có `1` tại đúng các vị trí là lần xuất hiện cuối (≤ `i`) của mỗi giá trị đã thấy.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Parity **chính xác** bằng mặt nạ bit (số nguyên lớn của Python) + đếm phân biệt trực tiếp

Dùng `mask ^= 1 << v` — Python có số nguyên lớn tuỳ ý, nên mặt nạ **không** sai xác suất. Đếm phân biệt bằng `len(set(nums[l:r+1]))`. Chỉ khả thi khi giá trị nhỏ (bộ nhớ `n × V / 8` byte), nhưng cho ta một lời giải **đúng tuyệt đối** để kiểm tra việc băm.

In [9]:
def valid_subarrays_exact_mask(nums, k, queries):
    P = [0]
    for x in nums:
        P.append(P[-1] ^ (1 << x))
    ans = []
    for l, r in queries:
        if P[r + 1] != P[l]:
            ans.append(False)
        else:
            ans.append(len(set(nums[l:r + 1])) == k)
    return ans

for nums, k, qs, exp in EXAMPLES:
    print(valid_subarrays_exact_mask(nums, k, qs), exp)

[False, True, False] [False, True, False]
[True, False] [True, False]


**Độ phức tạp.** Dựng mask: `O(n · V / 64)`; mỗi truy vấn chẵn tốn thêm `O(r - l)` để đếm phân biệt. Vẫn có thể `O(n q)` nếu nhiều truy vấn chẵn dài. Bộ nhớ `O(n · V / 8)` byte.

### 7.2. Cách 3 — Hash 64 bit + đếm phân biệt trực tiếp

Thay mặt nạ bằng số ngẫu nhiên 64 bit (bộ nhớ `O(n)`), parity kiểm tra `O(1)`. Phần đếm phân biệt vẫn trực tiếp: chỉ chạy cho truy vấn có parity chẵn.

In [10]:
def valid_subarrays_hash_direct(nums, k, queries, seed=2024):
    rng = random.Random(seed)
    h = {}
    P = [0]
    for x in nums:
        if x not in h:
            h[x] = rng.getrandbits(64)
        P.append(P[-1] ^ h[x])
    ans = []
    for l, r in queries:
        ans.append(P[r + 1] == P[l] and len(set(nums[l:r + 1])) == k)
    return ans

for nums, k, qs, exp in EXAMPLES:
    print(valid_subarrays_hash_direct(nums, k, qs), exp)

[False, True, False] [False, True, False]
[True, False] [True, False]


**Độ phức tạp.** `O(n + q)` cho parity; đếm phân biệt trực tiếp vẫn xấu `O(q · n)` nếu rất nhiều truy vấn chẵn và dài (ví dụ `nums` gồm `a a a a ...`, mọi đoạn độ dài chẵn đều chẵn).

### 7.3. Cách 4 — Hash 64 bit + BIT offline (bản nộp)

**Giải thích từng đoạn.**

1. Dựng `P` bằng hash ngẫu nhiên.
2. Gom truy vấn theo `r`: `by_r[r].append((l, idx))`.
3. Quét `i = 0..n-1`: cập nhật BIT (xóa `last`, thêm `i`). Với mỗi truy vấn kết thúc tại `i`: nếu `P[i+1] == P[l]` thì tính `distinct = prefix(i) - prefix(l - 1)` và so với `k`; ngược lại là sai.
4. Trả `ans` theo đúng thứ tự truy vấn ban đầu (dùng `idx`).

In [11]:
def valid_subarrays_hash_bit(nums, k, queries, seed=2024):
    n = len(nums)
    rng = random.Random(seed)
    h = {}
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        if x not in h:
            h[x] = rng.getrandbits(64)
        P[i + 1] = P[i] ^ h[x]

    by_r = [[] for _ in range(n)]
    for idx, (l, r) in enumerate(queries):
        by_r[r].append((l, idx))

    bit = Fenwick(n)
    last = {}
    ans = [False] * len(queries)
    for i, x in enumerate(nums):
        if x in last:
            bit.add(last[x], -1)
        bit.add(i, 1)
        last[x] = i
        for l, idx in by_r[i]:
            if P[i + 1] == P[l]:
                ans[idx] = (bit.prefix(i) - bit.prefix(l - 1) == k)
    return ans

for nums, k, qs, exp in EXAMPLES:
    print(valid_subarrays_hash_bit(nums, k, qs), exp)

[False, True, False] [False, True, False]
[True, False] [True, False]


**Độ phức tạp.** Dựng `P`: `O(n)`. Quét: mỗi `i` hai lần `add` (`O(log n)`), mỗi truy vấn chẵn hai lần `prefix` (`O(log n)`). Tổng `O((n + q) log n)`, bộ nhớ `O(n + q)`.

## 8. TRACE từng bước

Ta trace Cách 4 trên ví dụ 1: `nums = [1,2,2,1]`, `k = 2`, `queries = [[0,1],[0,3],[1,2]]`. Dùng hash là mặt nạ bit (`1 << v`) để dễ đọc; logic y hệt.

| i | x | `last` trước | BIT mark sau khi cập nhật | truy vấn kết thúc ở i | parity | distinct | kết quả |
|---|---|---|---|---|---|---|---|
| 0 | 1 | {} | `[1,0,0,0]` | — | — | — | — |
| 1 | 2 | {1:0} | `[1,1,0,0]` | `[0,1]` | P[2]=6≠P[0]=0 | — | sai |
| 2 | 2 | {1:0,2:1} | `[1,0,1,0]` (xoá vị trí 1, thêm 2) | `[1,2]` | P[3]=2=P[1]=2 | mark[1..2]=1 | 1≠2 → sai |
| 3 | 1 | {1:0,2:2} | `[0,0,1,1]` (xoá 0, thêm 3) | `[0,3]` | P[4]=0=P[0]=0 | mark[0..3]=2 | 2=2 → **đúng** |

In [12]:
def trace_valid_subarrays(nums, k, queries):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] ^ (1 << x))             # hash dễ đọc: mặt nạ bit
    by_r = [[] for _ in range(n)]
    for idx, (l, r) in enumerate(queries):
        by_r[r].append((l, idx))
    bit = Fenwick(n)
    last = {}
    ans = [False] * len(queries)
    print("P =", P)
    for i, x in enumerate(nums):
        print(f"\ni={i}, x={x}, last trước={last}")
        if x in last:
            bit.add(last[x], -1)
            print(f"   xoá mark ở vị trí {last[x]}")
        bit.add(i, 1)
        last[x] = i
        mark = [bit.prefix(j) - bit.prefix(j - 1) for j in range(n)]
        print("   mark =", mark)
        for l, idx in by_r[i]:
            even = P[i + 1] == P[l]
            if even:
                d = bit.prefix(i) - bit.prefix(l - 1)
                ans[idx] = (d == k)
                print(f"   truy vấn {idx} [{l},{i}]: parity chẵn, phân biệt={d}, k={k} -> {ans[idx]}")
            else:
                print(f"   truy vấn {idx} [{l},{i}]: parity LẺ (P[{i+1}]={P[i+1]} != P[{l}]={P[l]}) -> False")
    print("\nans =", ans)
    return ans

trace_valid_subarrays(*EXAMPLES[0][:3])

P = [0, 2, 6, 2, 0]

i=0, x=1, last trước={}
   mark = [1, 0, 0, 0]

i=1, x=2, last trước={1: 0}
   mark = [1, 1, 0, 0]
   truy vấn 0 [0,1]: parity LẺ (P[2]=6 != P[0]=0) -> False

i=2, x=2, last trước={1: 0, 2: 1}
   xoá mark ở vị trí 1
   mark = [1, 0, 1, 0]
   truy vấn 2 [1,2]: parity chẵn, phân biệt=1, k=2 -> False

i=3, x=1, last trước={1: 0, 2: 2}
   xoá mark ở vị trí 0
   mark = [0, 0, 1, 1]
   truy vấn 1 [0,3]: parity chẵn, phân biệt=2, k=2 -> True

ans = [False, True, False]


[False, True, False]

### 8.2. Trace ví dụ 2 (`[3,3,3]`, `k = 1`)

Quan sát: đoạn `[1,2] = [3,3]` có parity chẵn và đúng 1 phân biệt → đúng; đoạn `[0,2]` ba số 3 → parity lẻ → sai ngay mà không cần đếm.

In [13]:
trace_valid_subarrays(*EXAMPLES[1][:3])

P = [0, 8, 0, 8]

i=0, x=3, last trước={}
   mark = [1, 0, 0]

i=1, x=3, last trước={3: 0}
   xoá mark ở vị trí 0
   mark = [0, 1, 0]

i=2, x=3, last trước={3: 1}
   xoá mark ở vị trí 1
   mark = [0, 0, 1]
   truy vấn 0 [1,2]: parity chẵn, phân biệt=1, k=1 -> True
   truy vấn 1 [0,2]: parity LẺ (P[3]=8 != P[0]=0) -> False

ans = [True, False]


[True, False]

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4. Fenwick tree viết ngay trong phương thức bằng hai hàm lồng nhau để tiện dán lên LeetCode. Hash dùng `random.getrandbits(64)` (không cần seed cố định; xác suất sai ~ `q · 2^-64`).

In [14]:
import random as _random

class Solution:
    def validSubarrays(self, nums: list[int], k: int, queries: list[list[int]]) -> list[bool]:
        n = len(nums)
        # 1) Prefix XOR của hash ngẫu nhiên 64 bit -> kiểm tra "mọi tần suất chẵn" trong O(1)
        h = {}
        P = [0] * (n + 1)
        for i, x in enumerate(nums):
            if x not in h:
                h[x] = _random.getrandbits(64)
            P[i + 1] = P[i] ^ h[x]

        # 2) Gom truy vấn theo đầu mút phải
        by_r = [[] for _ in range(n)]
        for idx, (l, r) in enumerate(queries):
            by_r[r].append((l, idx))

        # 3) Fenwick tree đếm "lần xuất hiện cuối cùng" -> số giá trị phân biệt trong đoạn
        tree = [0] * (n + 1)

        def add(i, delta):
            i += 1
            while i <= n:
                tree[i] += delta
                i += i & -i

        def prefix(i):
            i += 1
            s = 0
            while i > 0:
                s += tree[i]
                i -= i & -i
            return s

        last = {}
        ans = [False] * len(queries)
        for i, x in enumerate(nums):
            if x in last:
                add(last[x], -1)
            add(i, 1)
            last[x] = i
            for l, idx in by_r[i]:
                if P[i + 1] == P[l] and prefix(i) - prefix(l - 1) == k:
                    ans[idx] = True
        return ans


sol = Solution()
for nums, k, qs, exp in EXAMPLES:
    assert sol.validSubarrays(nums, k, qs) == exp, (nums, k, qs)
print("2 ví dụ của đề: OK")

2 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Phần parity.** Gọi `S(v)` là số lần giá trị `v` xuất hiện trong `[l, r]`. Khi đó `P[r+1] ^ P[l] = XOR_{v : S(v) lẻ} h[v]` (mỗi giá trị chẵn tự triệt tiêu). Nếu mọi `S(v)` chẵn thì kết quả là `0` — nên `P[r+1] == P[l]` **luôn** đúng với đoạn hợp lệ (không bao giờ bỏ sót). Nếu tập `{v : S(v) lẻ}` khác rỗng, XOR là một tổ hợp ngẫu nhiên đều của các `h[v]` độc lập ⇒ bằng 0 với xác suất `2^-64`. Vậy **sai số một chiều** (có thể nhận nhầm đoạn lẻ là chẵn) cực nhỏ.

**(b) Phần đếm phân biệt.** Sau khi xử lý `i`, mỗi giá trị đã xuất hiện có đúng một vị trí được đánh dấu `1`: lần xuất hiện cuối cùng của nó (`≤ i`). Với `l ≤ i`, giá trị `v` có xuất hiện trong `[l, i]` ⇔ lần xuất hiện cuối của `v` (≤ i) nằm trong `[l, i]`. Vậy tổng đánh dấu trong `[l, i]` đúng bằng số giá trị phân biệt trong `[l, i]`.

**(c) Offline.** Sắp theo `r` không đổi đáp án vì mỗi truy vấn độc lập; ta lưu `idx` để trả đúng thứ tự.

**(d) Xác suất sai tổng.** Union bound: `≤ q · 2^-64 ≈ 10^5 / 1.8·10^19 ≈ 5·10^-15`. Nếu muốn an toàn hơn, dùng hash 128 bit (số nguyên Python lớn).

## 11. Phân tích Time/Space

Đặt `n = len(nums)`, `q = len(queries)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(q · n)` | `O(n)` |
| 2. Mặt nạ chính xác + `set` | `O(n·V/64 + q·n)` | `O(n·V/8)` byte |
| 3. Hash + `set` | `O(n + q·n)` trường hợp xấu | `O(n)` |
| 4. Hash + BIT offline | `O((n + q) log n)` | `O(n + q)` |

Cách 4 là cách duy nhất đạt `10^5` an toàn. Ta đo thực tế bên dưới.

In [15]:
import time
rng = random.Random(4033)
n = q = 100000
big = [rng.randint(1, 50) for _ in range(n)]
qs = []
for _ in range(q):
    l = rng.randint(0, n - 2); r = rng.randint(l + 1, n - 1)
    qs.append([l, r])
t = time.time(); res = Solution().validSubarrays(big, 3, qs); print(f"n=q=1e5 ngẫu nhiên: {time.time() - t:.2f}s, true = {sum(res)}")

# Trường hợp có nhiều truy vấn hợp lệ: toàn cặp liền nhau
pairs = [v for x in range(1, 50001) for v in (x, x)]
qs2 = [[2 * i, 2 * i + 1] for i in range(50000)] + [[0, 2 * i + 1] for i in range(50000)]
t = time.time(); res2 = Solution().validSubarrays(pairs, 1, qs2); print(f"n=1e5, q=1e5 nhiều đoạn chẵn: {time.time() - t:.2f}s, true = {sum(res2)}")

n=q=1e5 ngẫu nhiên: 0.26s, true = 0


n=1e5, q=1e5 nhiều đoạn chẵn: 0.28s, true = 50001


## 12. Bộ test

Ví dụ đề, edge case, rồi đối chiếu 4 cách (và `Solution`) trên input ngẫu nhiên có seed. Ta cố ý chọn **giá trị nhỏ** để nhiều đoạn chẵn xuất hiện, tránh test toàn `False`.

In [16]:
ALL = [valid_subarrays_bruteforce, valid_subarrays_exact_mask, valid_subarrays_hash_direct,
       valid_subarrays_hash_bit, Solution().validSubarrays]

EDGE = [
    ([1, 1], 1, [[0, 1]], [True]),
    ([1, 2], 2, [[0, 1]], [False]),
    ([1, 1], 2, [[0, 1]], [False]),                       # chẵn nhưng phân biệt 1 != 2
    ([1, 2, 2, 1], 2, [[0, 3]], [True]),
    ([1, 2, 1, 2], 2, [[0, 3], [0, 1], [1, 2], [2, 3]], [True, False, False, False]),
    ([5, 5, 5, 5], 1, [[0, 1], [0, 3], [1, 3], [0, 2]], [True, True, False, False]),
    ([1, 2, 3, 3, 2, 1], 3, [[0, 5], [1, 4], [2, 3]], [True, False, False]),
]
for fn in ALL:
    for nums, k, qs, exp in EXAMPLES + EDGE:
        assert fn(nums, k, qs) == exp, (fn.__name__, nums, k, qs, fn(nums, k, qs), exp)
print("Ví dụ đề + edge case: tất cả cách đúng")

rng = random.Random(403301)
trues = total = 0
for _ in range(1200):
    n = rng.randint(2, 14)
    nums = [rng.randint(1, 4) for _ in range(n)]
    k = rng.randint(1, 4)
    qs = []
    for _ in range(rng.randint(1, 12)):
        l = rng.randint(0, n - 2); r = rng.randint(l + 1, n - 1)
        qs.append([l, r])
    ref = valid_subarrays_bruteforce(nums, k, qs)
    trues += sum(ref); total += len(ref)
    for fn in ALL[1:]:
        assert fn(nums, k, qs) == ref, (fn.__name__, nums, k, qs)
print(f"1200 input ngẫu nhiên: mọi cách khớp vét cạn ({trues}/{total} truy vấn là True -> test có ý nghĩa)")
assert trues > 200

Ví dụ đề + edge case: tất cả cách đúng
1200 input ngẫu nhiên: mọi cách khớp vét cạn (298/7887 truy vấn là True -> test có ý nghĩa)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra "số phân biệt == k", quên điều kiện chẵn (hoặc ngược lại)

**Code sai:** `ans = (distinct == k)`.
**Vì sao sai:** ví dụ 1 truy vấn `[0,1] = [1,2]` có 2 phân biệt (= k) nhưng tần suất lẻ → phải sai.
**Code đúng:** `distinct == k and parity_even`.

In [17]:
def wrong_only_distinct(nums, k, queries):
    return [len(set(nums[l:r + 1])) == k for l, r in queries]

nums, k, qs, exp = EXAMPLES[0]
print("sai:", wrong_only_distinct(nums, k, qs), "| đúng:", exp)

sai: [True, True, False] | đúng: [False, True, False]


### Lỗi 2 — Đếm "tổng" số phần tử thay vì số phân biệt / quên xoá lần xuất hiện cũ trong BIT

**Code sai:** chỉ `bit.add(i, 1)` mà không `bit.add(last[x], -1)`.
**Vì sao sai:** BIT đếm **số phần tử** trong đoạn chứ không phải số giá trị phân biệt.
**Code đúng:** xoá mark cũ trước khi thêm mark mới.

In [18]:
def wrong_no_remove(nums, k, queries, seed=1):
    n = len(nums)
    rng = random.Random(seed)
    h = {}
    P = [0]
    for x in nums:
        h.setdefault(x, rng.getrandbits(64)); P.append(P[-1] ^ h[x])
    bit = Fenwick(n)
    by_r = [[] for _ in range(n)]
    for idx, (l, r) in enumerate(queries):
        by_r[r].append((l, idx))
    ans = [False] * len(queries)
    for i, x in enumerate(nums):
        bit.add(i, 1)                                  # SAI: không xoá lần cuối cũ
        for l, idx in by_r[i]:
            ans[idx] = P[i + 1] == P[l] and bit.prefix(i) - bit.prefix(l - 1) == k
    return ans

nums, k, qs, exp = EXAMPLES[0]
print("sai:", wrong_no_remove(nums, k, qs), "| đúng:", exp)       # [0,3] đếm 4 phần tử != 2

sai: [False, False, True] | đúng: [False, True, False]


### Lỗi 3 — Prefix BIT sai ở `l = 0`

**Code sai:** `bit.prefix(i) - bit.prefix(l - 1)` nhưng `prefix(-1)` bị cài đặt thành `prefix(n-1)` (chỉ số âm của Python) hoặc nhảy lỗi.
**Vì sao sai:** trong list Python chỉ số `-1` là phần tử cuối.
**Code đúng:** chuyển sang 1-based **trước khi** dùng (`i += 1` rồi `while i > 0`), nên `prefix(-1)` ra `0`.

### Lỗi 4 — Hash yếu (dùng chính giá trị `v`, hay `v * c`)

**Code sai:** `P[i+1] = P[i] ^ nums[i]`.
**Vì sao sai:** XOR của các giá trị khác nhau có thể bằng 0 (ví dụ `1 ^ 2 ^ 3 = 0`) dù mỗi giá trị xuất hiện **lẻ lần**.
**Code đúng:** gán mỗi giá trị một số **ngẫu nhiên** độc lập.

In [19]:
def wrong_xor_plain(nums, k, queries):
    P = [0]
    for x in nums:
        P.append(P[-1] ^ x)
    return [P[r + 1] == P[l] and len(set(nums[l:r + 1])) == k for l, r in queries]

nums = [1, 2, 3]
print("đoạn [1,2,3] có mọi tần suất chẵn?  XOR thường:", (1 ^ 2 ^ 3) == 0, "| thực tế: không")
print("sai:", wrong_xor_plain(nums, 3, [[0, 2]]), "| đúng:", valid_subarrays_bruteforce(nums, 3, [[0, 2]]))

đoạn [1,2,3] có mọi tần suất chẵn?  XOR thường: True | thực tế: không
sai: [True] | đúng: [False]


## 14. Mẹo & nhận dạng pattern

- **"Mọi phần tử xuất hiện chẵn lần trong đoạn"** → prefix XOR (mặt nạ) hoặc hash ngẫu nhiên; so hai prefix bằng nhau.
- **"Đếm số giá trị phân biệt trong đoạn", nhiều truy vấn** → offline theo `r` + BIT + "lần xuất hiện cuối" (hoặc Mo's algorithm `O((n+q)√n)`, hoặc cây phân đoạn persistent).
- Số nguyên lớn của Python cho phép "mặt nạ chính xác" để **kiểm chứng** cách dùng hash.
- Hash ngẫu nhiên + XOR là công cụ phổ biến để biến điều kiện đa chiều ("mọi giá trị chẵn") thành một phép so sánh duy nhất.
- Khi test cách dùng xác suất, hãy cố định seed trong notebook để kết quả lặp lại.
- Lọc điều kiện rẻ trước (parity `O(1)`), điều kiện đắt sau (BIT).

## 15. Biến thể

1. **Đếm số đoạn con mọi tần suất chẵn** (không có truy vấn): đếm cặp prefix hash bằng nhau — Bài 5.
2. **Mọi tần suất chia hết cho 3:** dùng cộng modulo 3 theo từng giá trị (tổng hash theo mod) thay cho XOR.
3. **Có đúng `k` giá trị lẻ tần suất:** `popcount` của mặt nạ chính xác (cần mặt nạ bit thật chứ không phải hash).
4. **Tần suất bằng nhau cho mọi giá trị trong đoạn:** dùng chuẩn hoá chênh lệch tần suất giữa các giá trị.
5. **Số phân biệt ≤ k** thay vì đúng `k`: đổi điều kiện cuối thành `<= k`.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | `Counter` từng đoạn | `O(q·n)` | `O(n)` | Trọng tài |
| 2. Mặt nạ chính xác | `1 << v` + `set` | `O(n·V/64 + q·n)` | `O(n·V/8)` | Kiểm chứng hash, giá trị nhỏ |
| 3. Hash + `set` | XOR hash, đếm trực tiếp | `O(n + q·n)` xấu | `O(n)` | Khi đoạn ngắn |
| 4. Hash + BIT offline | XOR hash + BIT lần cuối | `O((n+q) log n)` | `O(n+q)` | **Bản nộp** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** Với mặt nạ bit `1 << v`, tính mảng prefix `P` của `nums = [3, 1, 3, 1]` (dạng số nguyên). Các đoạn nào mọi tần suất chẵn?

**Bài 2 (tính tay).** `nums = [1,1,2,2,1,1]`, `k = 2`, `queries = [[0,3],[2,3],[1,2]]`. Trả lời từng truy vấn, giải thích.

**Bài 3 (điền TODO).** Hoàn thiện lớp `MyFenwick` với `add` và `prefix` (0-based bên ngoài).

**Bài 4 (tự viết lại).** Viết `my_valid_subarrays(nums, k, queries)` theo Cách 3 (hash + `set`).

**Bài 5 (mini challenge).** Viết `count_even_subarrays(nums)`: đếm số đoạn con `nums[l..r]` (độ dài ≥ 1) mà mọi tần suất chẵn — trong `O(n)` bằng hash + `Counter` các prefix.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
class MyFenwick:
    def __init__(self, n):
        # TODO: lưu n và tạo mảng t độ dài n + 1 toàn 0
        pass

    def add(self, i, delta):
        # TODO: i += 1; while i <= n: t[i] += delta; i += i & -i
        pass

    def prefix(self, i):
        # TODO: i += 1; s = 0; while i > 0: s += t[i]; i -= i & -i; return s
        pass

# Bài 4 — TODO
def my_valid_subarrays(nums, k, queries):
    # TODO: dựng prefix hash, rồi với mỗi truy vấn kiểm tra parity và len(set(...)) == k
    pass

# Bài 5 — TODO
def count_even_subarrays(nums):
    # TODO: prefix hash; đếm cặp (i < j) có P[i] == P[j] bằng Counter
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `1 << 3 = 8`, `1 << 1 = 2`.
- Mức 2: XOR dồn: `0, 8, 8^2=10, 10^8=2, 2^2=0`.
- Mức 3: `P = [0, 8, 10, 2, 0]`. Đoạn `[l, r]` chẵn ⇔ `P[r+1] == P[l]`: các cặp `(P[0], P[4])`: đoạn `[0,3]`. Cũng không có cặp prefix trùng nào khác (0 chỉ ở chỉ số 0 và 4; 8, 10, 2 xuất hiện một lần).

**Bài 2.**
- Mức 1: `[0,3] = [1,1,2,2]`, `[2,3] = [2,2]`, `[1,2] = [1,2]`.
- Mức 2: `[0,3]`: chẵn, phân biệt 2 = k. `[2,3]`: chẵn, phân biệt 1 ≠ 2. `[1,2]`: lẻ.
- Mức 3: đáp án `[True, False, False]`.

**Bài 3.**
- Mức 1: xem lớp `Fenwick` ở mục 3.5.
- Mức 2: chuyển sang 1-based bằng `i += 1`.
- Mức 3: `add` lặp `i += i & -i`; `prefix` lặp `i -= i & -i`.

**Bài 4.**
- Mức 1: xem Cách 3.
- Mức 2: `random.Random(seed).getrandbits(64)` cho mỗi giá trị mới.
- Mức 3: `ans.append(P[r+1] == P[l] and len(set(nums[l:r+1])) == k)`.

**Bài 5.**
- Mức 1: đoạn `(i, j]` chẵn ⇔ `P[i] == P[j]`.
- Mức 2: duyệt `j`, cộng `cnt[P[j]]` rồi tăng `cnt[P[j]]`.
- Mức 3: bắt đầu với `cnt = Counter({0: 1})` (prefix rỗng).

## 19. Đáp án tham khảo

In [21]:
# Bài 1
P = [0]
for x in [3, 1, 3, 1]:
    P.append(P[-1] ^ (1 << x))
assert P == [0, 8, 10, 2, 0]

# Bài 2
assert Solution().validSubarrays([1, 1, 2, 2, 1, 1], 2, [[0, 3], [2, 3], [1, 2]]) == [True, False, False]

# Bài 3
class MyFenwickAns:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)
    def add(self, i, delta):
        i += 1
        while i <= self.n:
            self.t[i] += delta
            i += i & -i
    def prefix(self, i):
        i += 1
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

f1, f2 = Fenwick(20), MyFenwickAns(20)
rr = random.Random(5)
for _ in range(200):
    p, d = rr.randint(0, 19), rr.randint(-3, 3)
    f1.add(p, d); f2.add(p, d)
    q_ = rr.randint(-1, 19)
    assert f1.prefix(q_) == f2.prefix(q_)
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4
def my_valid_subarrays_ans(nums, k, queries):
    rng = random.Random(7)
    h, P = {}, [0]
    for x in nums:
        if x not in h:
            h[x] = rng.getrandbits(64)
        P.append(P[-1] ^ h[x])
    return [P[r + 1] == P[l] and len(set(nums[l:r + 1])) == k for l, r in queries]

# Bài 5
def count_even_subarrays_ans(nums):
    rng = random.Random(11)
    h = {}
    cnt = Counter({0: 1})
    cur = total = 0
    for x in nums:
        if x not in h:
            h[x] = rng.getrandbits(64)
        cur ^= h[x]
        total += cnt[cur]
        cnt[cur] += 1
    return total

def count_even_bruteforce(nums):
    n, t = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            if all(v % 2 == 0 for v in Counter(nums[l:r + 1]).values()):
                t += 1
    return t

rng = random.Random(99)
for _ in range(300):
    n = rng.randint(1, 12)
    nums = [rng.randint(1, 3) for _ in range(n)]
    qs = [[l, r] for l in range(n) for r in range(l + 1, n)]
    k = rng.randint(1, 3)
    if qs:
        assert my_valid_subarrays_ans(nums, k, qs) == valid_subarrays_bruteforce(nums, k, qs)
    assert count_even_subarrays_ans(nums) == count_even_bruteforce(nums)
assert count_even_subarrays_ans([1, 2, 2, 1]) == 2     # [2,2] và [1,2,2,1]
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
# 1) parity: hash ngẫu nhiên 64 bit cho mỗi giá trị, prefix XOR
P[i+1] = P[i] ^ h[nums[i]]
even(l, r) <=> P[r+1] == P[l]

# 2) phân biệt trong đoạn: offline theo r + BIT "lần xuất hiện cuối"
for i, x in enumerate(nums):
    if x in last: bit.add(last[x], -1)
    bit.add(i, 1); last[x] = i
    for (l, idx) in by_r[i]:
        distinct = bit.prefix(i) - bit.prefix(l - 1)

ans[idx] = even(l, i) and distinct == k
```

**Checklist trước khi submit:**

- [ ] Cả hai điều kiện: tần suất chẵn **và** đúng `k` phân biệt.
- [ ] Hash ngẫu nhiên độc lập cho mỗi giá trị (không dùng XOR thẳng `nums[i]`).
- [ ] BIT: xoá mark cũ trước khi thêm mark mới; `prefix(-1) = 0`.
- [ ] Gom truy vấn theo `r`, lưu `idx` để trả đúng thứ tự.
- [ ] Giải thích được xác suất sai (`q · 2^-64`) nếu được hỏi.
- [ ] Đối chiếu với vét cạn trên input ngẫu nhiên nhỏ có nhiều đoạn chẵn.